In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np
import torch
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from pathlib import Path

DEVICE      = torch.device('cuda')
OUTPUT_PATH = Path('enalis_3class')
CLASS_NAMES = ['at_risk', 'fresh', 'rotten']

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

test_ds     = datasets.ImageFolder(OUTPUT_PATH / 'test', transform=eval_transforms)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                         num_workers=2, pin_memory=True)

# ── Inference function — returns predictions and ground-truth labels ───────
@torch.no_grad()
def get_preds(model):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in test_loader:
        images = images.to(DEVICE)
        out    = model(images)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)

# ── Load the three model checkpoints ──────────────────────────────────────

# Xception — reload architecture and trained weights
xception = timm.create_model('xception', pretrained=False, num_classes=3).to(DEVICE)
xception.load_state_dict(torch.load('Xception_enalis_best.pth', map_location=DEVICE))

# Swin Transformer — reload architecture and trained weights
swin = timm.create_model('swin_tiny_patch4_window7_224',
                          pretrained=False, num_classes=3).to(DEVICE)
swin.load_state_dict(torch.load('SwinTransformer_enalis_best.pth', map_location=DEVICE))

# YOLOv5m — loaded from Ultralytics checkpoint format
ckpt  = torch.load('runs/enalis_yolo/yolov5m_enalis/weights/best.pt',
                    map_location=DEVICE)
yolo  = ckpt['model'].float().to(DEVICE)

# ── Run inference for all three models ────────────────────────────────────
preds_swin, labels = get_preds(swin)
preds_yolo, _      = get_preds(yolo)
preds_xcep, _      = get_preds(xception)

# ── McNemar's test — pairwise comparison ──────────────────────────────────
# b = samples correct for model A but wrong for model B
# c = samples correct for model B but wrong for model A
# A significant result (p < 0.05) means the two models disagree
# systematically, not just randomly
def mcnemar_test(preds_a, preds_b, labels):
    correct_a = (preds_a == labels)
    correct_b = (preds_b == labels)
    b = int(np.sum( correct_a & ~correct_b))
    c = int(np.sum(~correct_a &  correct_b))
    table  = [[0, b], [c, 0]]
    result = mcnemar(table, exact=False, correction=True)
    return b, c, result.pvalue

# ── Define pairs to compare ───────────────────────────────────────────────
pairs = [
    ('Swin',    preds_swin, 'YOLOv5m',  preds_yolo),
    ('Swin',    preds_swin, 'Xception', preds_xcep),
    ('YOLOv5m', preds_yolo, 'Xception', preds_xcep),
]

# ── Print results table ───────────────────────────────────────────────────
print(f"{'Model A':<15} {'Model B':<12} {'b':>5} {'c':>5} {'p-value':>12}")
print("-" * 55)
for name_a, pa, name_b, pb in pairs:
    b, c, p = mcnemar_test(pa, pb, labels)
    sig = "***" if p < 0.001 else ("*" if p < 0.05 else "n.s.")
    print(f"{name_a:<15} {name_b:<12} {b:>5} {c:>5} {p:>12.4f}  {sig}")